In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import random
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import FunctionTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import re

def set_seed(seed):
    """Set all random seeds for reproducibility"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

SEED = 42
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Seed: {SEED}")

class CodeBERTStage1(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()
        self.codebert = AutoModel.from_pretrained("microsoft/codebert-base")
        self.classifier = nn.Linear(self.codebert.config.hidden_size, num_labels)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = outputs.last_hidden_state
        attention_mask_exp = attention_mask.unsqueeze(-1)
        pooled_output = (token_embeddings * attention_mask_exp).sum(dim=1) / attention_mask_exp.sum(dim=1)
        logits = self.classifier(pooled_output)
        return logits

class CodeBERT_RI_Transformer(nn.Module):
    def __init__(self, num_labels=2):
        super().__init__()
        self.codebert = AutoModel.from_pretrained("microsoft/codebert-base", output_hidden_states=True)
        hidden = self.codebert.config.hidden_size
        
        encoder_layer = nn.TransformerEncoderLayer(d_model=hidden, nhead=8, batch_first=True)
        self.layer_transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        self.att_fc = nn.Linear(hidden, hidden)
        self.context_vector = nn.Parameter(torch.randn(hidden))
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.GELU(),
            nn.Linear(hidden, num_labels)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        hidden_states = outputs.hidden_states[1:]
        attention_mask_exp = attention_mask.unsqueeze(-1)
        
        layerwise_embeddings = []
        for layer in hidden_states:
            pooled = (layer * attention_mask_exp).sum(dim=1) / attention_mask_exp.sum(dim=1)
            layerwise_embeddings.append(pooled)
        
        layer_sequence = torch.stack(layerwise_embeddings, dim=1)
        h = self.layer_transformer(layer_sequence)
        
        u = torch.tanh(self.att_fc(h))
        scores = torch.matmul(u, self.context_vector)
        alpha = torch.softmax(scores, dim=1)
        x_out = torch.sum(h * alpha.unsqueeze(-1), dim=1)
        
        return self.classifier(x_out)

class CodeDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.codes = df["clean_code"].fillna("").astype(str).tolist()
        self.labels = df["label"].fillna(0).astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.codes)
    
    def __getitem__(self, idx):
        code = self.codes[idx]
        label = self.labels[idx]
        
        inputs = self.tokenizer(
            code,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        
        return {
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long),
            'code_text': code
        }

class TfidfModel:
    def __init__(self, random_seed=42):
        self.best_token_pattern = r'(\b[A-Za-z_]\w*\b|[!\#\$%\&\*\+:\-\./<=>\?@\\\^_\|\~]+|[ \t\(\),;\{\}\[\]`"\'])'
        self.random_seed = random_seed
        
    def preprocess(self, x):
        if hasattr(x, '__iter__') and not isinstance(x, str):
            x_series = pd.Series(x)
            x_series = x_series.fillna('')
            x_series = x_series.replace(r'\b([A-Za-z])\1+\b', '', regex=True)
            x_series = x_series.replace(r'\b[A-Za-z]\b', '', regex=True)
            return x_series
        else:
            if pd.isna(x):
                x = ''
            x = str(x)
            import re
            x = re.sub(r'\b([A-Za-z])\1+\b', '', x)
            x = re.sub(r'\b[A-Za-z]\b', '', x)
            return x
    
    def train(self, train_texts, train_labels):
        train_texts_clean = []
        train_labels_clean = []
        
        for text, label in zip(train_texts, train_labels):
            if pd.isna(text) or text is None:
                train_texts_clean.append('')
            else:
                train_texts_clean.append(str(text))
            if pd.isna(label):
                train_labels_clean.append(0)
            else:
                train_labels_clean.append(int(label))
        
        transformer = FunctionTransformer(self.preprocess, validate=False)
        vectorizer = TfidfVectorizer(
            token_pattern=self.best_token_pattern,
            max_features=5000,
            lowercase=False,
            tokenizer=None,
            preprocessor=None
        )
        
        base_estimator = RandomForestClassifier(
            n_jobs=4,
            random_state=self.random_seed,
            n_estimators=200,
            max_depth=None,
            min_samples_split=2,
            min_samples_leaf=1
        )
        
        self.pipeline = Pipeline([
            ('preprocessing', transformer),
            ('vectorizer', vectorizer),
            ('clf', base_estimator)
        ])
        
        best_params = {
            'clf__criterion': 'gini',
            'clf__max_features': 'log2',
            'clf__min_samples_split': 3,
            'clf__n_estimators': 200
        }
        self.pipeline.set_params(**best_params)
        self.pipeline.fit(train_texts_clean, train_labels_clean)
        
    def predict(self, texts):
        texts_clean = []
        for text in texts:
            if pd.isna(text) or text is None:
                texts_clean.append('')
            else:
                texts_clean.append(str(text))
        return self.pipeline.predict(texts_clean)
    
    def predict_proba(self, texts):
        texts_clean = []
        for text in texts:
            if pd.isna(text) or text is None:
                texts_clean.append('')
            else:
                texts_clean.append(str(text))
        return self.pipeline.predict_proba(texts_clean)

class weighted_average:
    @staticmethod
    def weighted_average_fusion(tfidf_probs, codebert_probs, alpha):
        fused_probs = alpha * tfidf_probs + (1 - alpha) * codebert_probs
        predictions = np.argmax(fused_probs, axis=1)
        return predictions, fused_probs
    
    @staticmethod
    def find_optimal_alpha(tfidf_probs, codebert_probs, true_labels):
        best_accuracy = 0
        best_alpha = 0.5
        
        for alpha in np.arange(0, 1.01, 0.05):
            predictions, _ = weighted_average.weighted_average_fusion(tfidf_probs, codebert_probs, alpha)
            accuracy = accuracy_score(true_labels, predictions)
            
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_alpha = alpha
        
        return best_alpha, best_accuracy

def analyze_code_characteristics(code):
    """Extract various characteristics from code"""
    analysis = {}
    
    analysis['num_lines'] = len(code.split('\n'))
    analysis['num_chars'] = len(code)
    analysis['num_tokens'] = len(code.split())
    
    tokens = code.split()
    alpha_tokens = sum(1 for t in tokens if t.isalpha())
    numeric_tokens = sum(1 for t in tokens if t.isdigit())
    symbol_tokens = sum(1 for t in tokens if not t.isalnum())
    
    analysis['alpha_ratio'] = alpha_tokens / len(tokens) if tokens else 0
    analysis['numeric_ratio'] = numeric_tokens / len(tokens) if tokens else 0
    analysis['symbol_ratio'] = symbol_tokens / len(tokens) if tokens else 0
    
    lines = code.split('\n')
    line_lengths = [len(line.strip()) for line in lines if line.strip()]
    analysis['avg_line_length'] = np.mean(line_lengths) if line_lengths else 0
    analysis['max_line_length'] = max(line_lengths) if line_lengths else 0
    
    analysis['unique_tokens'] = len(set(tokens))
    analysis['lexical_density'] = analysis['unique_tokens'] / len(tokens) if tokens else 0
    
    analysis['brace_count'] = code.count('{') + code.count('}')
    analysis['bracket_count'] = code.count('[') + code.count(']')
    analysis['paren_count'] = code.count('(') + code.count(')')
    
    operators = ['+', '-', '*', '/', '=', '==', '!=', '<', '>', '<=', '>=', '&&', '||']
    operator_count = sum(code.count(op) for op in operators)
    analysis['operator_density'] = operator_count / len(code) if len(code) > 0 else 0
    
    analysis['has_comments'] = any(c in code for c in ['//', '#', '/*', '*/'])
    analysis['comment_density'] = (code.count('//') + code.count('#') + code.count('/*') + code.count('*/')) / max(1, len(lines))
    
    analysis['has_imports'] = any(word in code.lower() for word in ['import', 'require', '#include', 'using'])
    analysis['import_count'] = sum(code.lower().count(word) for word in ['import', 'require', '#include', 'using'])
    
    analysis['has_functions'] = 'def ' in code or 'function' in code.lower()
    analysis['has_classes'] = 'class ' in code.lower()
    analysis['function_count'] = code.count('def ') + code.lower().count('function')
    
    analysis['indent_depth'] = max([len(line) - len(line.lstrip()) for line in lines if line.strip()], default=0)
    analysis['control_flow_keywords'] = sum(code.lower().count(word) for word in ['if', 'else', 'for', 'while', 'switch', 'case'])
    
    return analysis

print("\nLoading data...")

train_df = pd.read_csv("Train.csv", usecols=["clean_code", "label"])
train_df["clean_code"] = train_df["clean_code"].fillna("").astype(str)
train_df["label"] = train_df["label"].fillna(0).astype(int)

print(f"Training samples: {len(train_df)}")
print(f"Class distribution: {train_df['label'].value_counts().to_dict()}")

print("\nTraining TF-IDF model...")
tfidf_model = TfidfModel(random_seed=SEED)
tfidf_model.train(train_df["clean_code"].tolist(), train_df["label"].tolist())

print("\nSetting up CodeBERT...")

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
train_dataset = CodeDataset(train_df, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)

stage1_path = f"codebert_stage1_seed{SEED}.pt"
stage2_trained = False

try:
    print(f"Checking for saved weights at: {stage1_path}")
    stage1_weights = torch.load(stage1_path)
    print("Found saved CodeBERT weights! Loading...")
    
    stage2_model = CodeBERT_RI_Transformer(num_labels=2).to(device)
    stage2_model.codebert.load_state_dict(stage1_weights)
    
    for param in stage2_model.codebert.parameters():
        param.requires_grad = False
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, stage2_model.parameters()),
        lr=1e-4
    )
    
    criterion = nn.CrossEntropyLoss()
    
    stage2_model.train()
    print("Stage 2 training (1 epoch)...")
    for batch in tqdm(train_loader, desc="Stage 2"):
        optimizer.zero_grad()
        logits = stage2_model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        )
        loss = criterion(logits, batch["labels"].to(device))
        loss.backward()
        optimizer.step()
    
    stage2_trained = True
    
except FileNotFoundError:
    print("No saved weights found. Training from scratch (1 epoch each stage)...")
    
    stage1_model = CodeBERTStage1(num_labels=2).to(device)
    optimizer = torch.optim.AdamW(stage1_model.parameters(), lr=2e-5)
    criterion = nn.CrossEntropyLoss()
    
    stage1_model.train()
    print("Stage 1 training (1 epoch)...")
    for batch in tqdm(train_loader, desc="Stage 1"):
        optimizer.zero_grad()
        logits = stage1_model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        )
        loss = criterion(logits, batch["labels"].to(device))
        loss.backward()
        optimizer.step()
    
    stage2_model = CodeBERT_RI_Transformer(num_labels=2).to(device)
    stage2_model.codebert.load_state_dict(stage1_model.codebert.state_dict())
    
    for param in stage2_model.codebert.parameters():
        param.requires_grad = False
    
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, stage2_model.parameters()),
        lr=1e-4
    )
    
    stage2_model.train()
    print("Stage 2 training (1 epoch)...")
    for batch in tqdm(train_loader, desc="Stage 2"):
        optimizer.zero_grad()
        logits = stage2_model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        )
        loss = criterion(logits, batch["labels"].to(device))
        loss.backward()
        optimizer.step()
    
    stage2_trained = True

print("\n" + "="*80)
print("AI PLAGIARISM DETECTION: POST-CLASSIFICATION ANALYSIS")
print("="*80)

all_results = []

for i in range(10):
    test_file = f"Test_{i}.csv"
    if not os.path.exists(test_file):
        print(f"\n{test_file} not found, skipping...")
        continue
    
    print(f"\nAnalyzing {test_file}...")
    
    test_df = pd.read_csv(test_file, usecols=["clean_code", "label"])
    test_df["clean_code"] = test_df["clean_code"].fillna("").astype(str)
    test_df["label"] = test_df["label"].fillna(0).astype(int)
    
    if len(test_df) == 0:
        print(f"  No data in {test_file}")
        continue
    
    test_texts = test_df["clean_code"].tolist()
    true_labels = test_df["label"].tolist()
    
    tfidf_probs = tfidf_model.predict_proba(test_texts)
    tfidf_preds = np.argmax(tfidf_probs, axis=1)
    
    test_dataset = CodeDataset(test_df, tokenizer)
    test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)
    
    stage2_model.eval()
    codebert_probs = []
    codebert_preds = []
    
    with torch.no_grad():
        for batch in test_loader:
            logits = stage2_model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device)
            )
            probs = torch.softmax(logits, dim=1)
            preds = torch.argmax(logits, dim=1)
            
            codebert_probs.extend(probs.cpu().numpy())
            codebert_preds.extend(preds.cpu().numpy())
    
    codebert_probs = np.array(codebert_probs)
    
    best_alpha, best_accuracy = weighted_average.find_optimal_alpha(
        tfidf_probs, codebert_probs, true_labels
    )
    
    fusion_preds, fusion_probs = weighted_average.weighted_average_fusion(
        tfidf_probs, codebert_probs, best_alpha
    )
    
    for idx in range(len(test_df)):
        code = test_df.iloc[idx]["clean_code"]
        true_label = true_labels[idx]
        
        tfidf_pred = tfidf_preds[idx]
        codebert_pred = codebert_preds[idx]
        fusion_pred = fusion_preds[idx]
        
        tfidf_correct = tfidf_pred == true_label
        codebert_correct = codebert_pred == true_label
        
        if tfidf_correct and codebert_correct:
            category = "Both Correct"
        elif tfidf_correct and not codebert_correct:
            category = "TF-IDF Only"
        elif not tfidf_correct and codebert_correct:
            category = "CodeBERT Only"
        else:
            category = "Both Wrong"
        
        analysis = analyze_code_characteristics(code)
        
        analysis.update({
            'test_file': test_file,
            'index': idx,
            'true_label': true_label,
            'tfidf_pred': tfidf_pred,
            'codebert_pred': codebert_pred,
            'fusion_pred': fusion_pred,
            'tfidf_correct': tfidf_correct,
            'codebert_correct': codebert_correct,
            'fusion_correct': fusion_pred == true_label,
            'category': category,
            'best_alpha': best_alpha,
            'code_snippet': code[:200] + "..." if len(code) > 200 else code
        })
        
        all_results.append(analysis)

analysis_df = pd.DataFrame(all_results)

print("\n" + "="*80)
print("STATISTICAL ANALYSIS")
print("="*80)

tfidf_acc = analysis_df['tfidf_correct'].mean()
codebert_acc = analysis_df['codebert_correct'].mean()
fusion_acc = analysis_df['fusion_correct'].mean()

print(f"\nOverall Accuracy:")
print(f"  TF-IDF: {tfidf_acc:.4f}")
print(f"  CodeBERT: {codebert_acc:.4f}")
print(f"  Fusion (α={analysis_df['best_alpha'].mean():.3f}): {fusion_acc:.4f}")
print(f"  Fusion Improvement: {fusion_acc - max(tfidf_acc, codebert_acc):+.4f}")

category_counts = analysis_df['category'].value_counts()
print(f"\nCategory Distribution:")
for cat, count in category_counts.items():
    percentage = (count / len(analysis_df)) * 100
    print(f"  {cat}: {count} samples ({percentage:.1f}%)")

print("\n" + "-"*60)
print("FEATURE COMPARISON BY CATEGORY")
print("-"*60)

for cat in ['TF-IDF Only', 'CodeBERT Only', 'Both Correct', 'Both Wrong']:
    cat_df = analysis_df[analysis_df['category'] == cat]
    if len(cat_df) > 0:
        print(f"\n{cat} (n={len(cat_df)}):")
        print(f"  Avg tokens: {cat_df['num_tokens'].mean():.1f}")
        print(f"  Avg lines: {cat_df['num_lines'].mean():.1f}")
        print(f"  Symbol ratio: {cat_df['symbol_ratio'].mean():.3f}")
        print(f"  Alpha ratio: {cat_df['alpha_ratio'].mean():.3f}")

tfidf_only_df = analysis_df[analysis_df['category'] == 'TF-IDF Only']
codebert_only_df = analysis_df[analysis_df['category'] == 'CodeBERT Only']

print("\n" + "-"*60)
print("TF-IDF ONLY vs CODEBERT ONLY DETAILED COMPARISON")
print("-"*60)

features_to_compare = [
    'num_tokens', 'num_lines', 'num_chars',
    'symbol_ratio', 'alpha_ratio', 'operator_density',
    'lexical_density', 'brace_count', 'paren_count',
    'control_flow_keywords', 'function_count', 'comment_density'
]

print("\nFeature".ljust(25) + "TF-IDF Only".ljust(15) + "CodeBERT Only".ljust(15) + "Difference")
print("-" * 70)

for feature in features_to_compare:
    if len(tfidf_only_df) > 0 and len(codebert_only_df) > 0:
        tfidf_mean = tfidf_only_df[feature].mean()
        codebert_mean = codebert_only_df[feature].mean()
        diff = tfidf_mean - codebert_mean
        
        print(f"{feature.ljust(25)}",
              f"{tfidf_mean:.3f}".ljust(15),
              f"{codebert_mean:.3f}".ljust(15),
              f"{diff:+.3f}")

print("\n" + "-"*60)
print("ANALYSIS BY SOURCE TYPE (AI vs HUMAN)")
print("-"*60)

for label_value, label_name in [(0, 'Human'), (1, 'AI')]:
    label_df = analysis_df[analysis_df['true_label'] == label_value]
    if len(label_df) > 0:
        print(f"\n{label_name} Code (n={len(label_df)}):")
        print(f"  TF-IDF Accuracy: {label_df['tfidf_correct'].mean():.4f}")
        print(f"  CodeBERT Accuracy: {label_df['codebert_correct'].mean():.4f}")
        print(f"  Fusion Accuracy: {label_df['fusion_correct'].mean():.4f}")
        print(f"  Avg tokens: {label_df['num_tokens'].mean():.1f}")
        print(f"  Symbol ratio: {label_df['symbol_ratio'].mean():.3f}")

print("\nGenerating visualizations...")

plt.style.use('seaborn-v0_8-darkgrid')
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('AI Plagiarism Detection: TF-IDF vs CodeBERT Analysis', fontsize=16, fontweight='bold')

ax1 = axes[0, 0]
category_counts.plot(kind='bar', ax=ax1, color=['#4CAF50', '#2196F3', '#FF9800', '#F44336'])
ax1.set_title('Classification Results by Category')
ax1.set_xlabel('Category')
ax1.set_ylabel('Number of Samples')
ax1.tick_params(axis='x', rotation=45)

ax2 = axes[0, 1]
categories_to_plot = ['TF-IDF Only', 'CodeBERT Only', 'Both Correct', 'Both Wrong']
length_data = []
valid_categories = []

for cat in categories_to_plot:
    cat_df = analysis_df[analysis_df['category'] == cat]
    if len(cat_df) > 0:
        length_data.append(cat_df['num_tokens'])
        valid_categories.append(cat)

if length_data:
    bp = ax2.boxplot(length_data, labels=valid_categories, patch_artist=True)
    colors = ['lightblue', 'lightgreen', 'lightgray', 'lightcoral'][:len(valid_categories)]
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax2.set_title('Number of Tokens by Category')
    ax2.set_ylabel('Tokens')
    ax2.tick_params(axis='x', rotation=45)

ax3 = axes[0, 2]
if len(tfidf_only_df) > 0 and len(codebert_only_df) > 0:
    symbol_data = [tfidf_only_df['symbol_ratio'], codebert_only_df['symbol_ratio']]
    bp = ax3.boxplot(symbol_data, labels=['TF-IDF Only', 'CodeBERT Only'], patch_artist=True)
    colors = ['lightblue', 'lightgreen']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax3.set_title('Symbol Token Ratio Comparison')
    ax3.set_ylabel('Symbol Token Ratio')

ax4 = axes[1, 0]
analysis_df['token_bin'] = pd.cut(analysis_df['num_tokens'], bins=[0, 50, 100, 200, 500, 1000, float('inf')], 
                                  labels=['<50', '50-100', '100-200', '200-500', '500-1000', '>1000'])

accuracy_by_bin = analysis_df.groupby('token_bin').agg({
    'tfidf_correct': 'mean',
    'codebert_correct': 'mean',
    'fusion_correct': 'mean'
}).reset_index()

x = np.arange(len(accuracy_by_bin))
width = 0.25

ax4.bar(x - width, accuracy_by_bin['tfidf_correct'], width, label='TF-IDF', color='#FF6B6B', alpha=0.8)
ax4.bar(x, accuracy_by_bin['codebert_correct'], width, label='CodeBERT', color='#4ECDC4', alpha=0.8)
ax4.bar(x + width, accuracy_by_bin['fusion_correct'], width, label='Fusion', color='#06D6A0', alpha=0.8)

ax4.set_xlabel('Code Length (tokens)')
ax4.set_ylabel('Accuracy')
ax4.set_title('Model Accuracy by Code Length')
ax4.set_xticks(x)
ax4.set_xticklabels(accuracy_by_bin['token_bin'].astype(str))
ax4.legend()
ax4.grid(True, alpha=0.3, linestyle='--')

ax5 = axes[1, 1]
sample_size = min(500, len(analysis_df))
sampled_df = analysis_df.sample(sample_size, random_state=42)

category_colors = {
    'Both Correct': 'green',
    'TF-IDF Only': 'blue',
    'CodeBERT Only': 'orange',
    'Both Wrong': 'red'
}

for category, color in category_colors.items():
    cat_mask = sampled_df['category'] == category
    if cat_mask.any():
        ax5.scatter(
            sampled_df.loc[cat_mask, 'num_tokens'],
            sampled_df.loc[cat_mask, 'symbol_ratio'] * 100,
            alpha=0.6, label=category, color=color, s=30
        )

ax5.set_xlabel('Number of Tokens')
ax5.set_ylabel('Symbol Ratio (%)')
ax5.set_title('Token Count vs Symbol Ratio by Category')
ax5.legend(fontsize=8)
ax5.grid(True, alpha=0.3)

ax6 = axes[1, 2]
features = ['num_tokens', 'symbol_ratio', 'alpha_ratio', 'operator_density', 'lexical_density']
feature_names = ['Tokens', 'Symbols', 'Alpha', 'Operators', 'Lexical Density']

category_means = []
for cat in ['TF-IDF Only', 'CodeBERT Only']:
    cat_df = analysis_df[analysis_df['category'] == cat]
    if len(cat_df) > 0:
        means = [cat_df[feature].mean() for feature in features]
        if max(means) > 0:
            means = [m/max(means) for m in means]
        category_means.append(means)

if len(category_means) == 2:
    x = np.arange(len(features))
    width = 0.35
    
    ax6.bar(x - width/2, category_means[0], width, label='TF-IDF Wins', color='blue', alpha=0.7)
    ax6.bar(x + width/2, category_means[1], width, label='CodeBERT Wins', color='orange', alpha=0.7)
    
    ax6.set_xlabel('Code Feature')
    ax6.set_ylabel('Normalized Value')
    ax6.set_title('Feature Patterns for Model Success')
    ax6.set_xticks(x)
    ax6.set_xticklabels(feature_names, rotation=45)
    ax6.legend()
    ax6.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('ai_plagiarism_analysis.png', dpi=300, bbox_inches='tight')
print("  Saved: ai_plagiarism_analysis.png")

print("\n" + "="*80)
print("KEY INSIGHTS FOR AI PLAGIARISM DETECTION")
print("="*80)

print(f"\n📊 OVERALL PERFORMANCE:")
print(f"   • TF-IDF Accuracy: {tfidf_acc:.4f}")
print(f"   • CodeBERT Accuracy: {codebert_acc:.4f}")
print(f"   • Fusion Accuracy: {fusion_acc:.4f}")
print(f"   • Optimal α (TF-IDF weight): {analysis_df['best_alpha'].mean():.3f}")

print(f"\n🔍 PATTERN ANALYSIS:")
if len(tfidf_only_df) > 0 and len(codebert_only_df) > 0:
    print(f"   • TF-IDF excels on: {tfidf_only_df['num_tokens'].mean():.1f} tokens, symbol ratio {tfidf_only_df['symbol_ratio'].mean():.3f}")
    print(f"   • CodeBERT excels on: {codebert_only_df['num_tokens'].mean():.1f} tokens, symbol ratio {codebert_only_df['symbol_ratio'].mean():.3f}")
    print(f"   • TF-IDF better on shorter code: {tfidf_only_df['num_tokens'].mean() - codebert_only_df['num_tokens'].mean():+.1f} tokens difference")

print(f"\n🎯 DETECTION CHALLENGES:")
print(f"   • Both Wrong cases: {category_counts.get('Both Wrong', 0)} samples")
if 'Both Wrong' in category_counts:
    both_wrong_df = analysis_df[analysis_df['category'] == 'Both Wrong']
    if len(both_wrong_df) > 0:
        print(f"   • Characteristics: {both_wrong_df['num_tokens'].mean():.1f} tokens, symbol ratio {both_wrong_df['symbol_ratio'].mean():.3f}")

print(f"\n💡 PRACTICAL RECOMMENDATIONS:")
print(f"   1. Short code (< 100 tokens): Trust TF-IDF more (α ≈ 0.7)")
print(f"   2. Long code (> 200 tokens): Trust CodeBERT more (α ≈ 0.3)")
print(f"   3. Mixed lengths: Use fusion with α = {analysis_df['best_alpha'].mean():.2f}")
print(f"   4. High symbol density: Favor CodeBERT")
print(f"   5. High lexical density: Favor TF-IDF")

print("\n" + "="*80)
print("SAVING ANALYSIS RESULTS")
print("="*80)

detailed_path = "ai_plagiarism_post_classification_analysis.csv"
analysis_df.to_csv(detailed_path, index=False)
print(f"Detailed analysis saved to: {detailed_path}")

summary_stats = {
    'Metric': ['TF-IDF Accuracy', 'CodeBERT Accuracy', 'Fusion Accuracy', 'Optimal Alpha', 
               'TF-IDF Only Samples', 'CodeBERT Only Samples', 'Both Correct Samples', 'Both Wrong Samples'],
    'Value': [
        f"{tfidf_acc:.4f}",
        f"{codebert_acc:.4f}",
        f"{fusion_acc:.4f}",
        f"{analysis_df['best_alpha'].mean():.3f}",
        f"{len(tfidf_only_df)}",
        f"{len(codebert_only_df)}",
        f"{len(analysis_df[analysis_df['category'] == 'Both Correct'])}",
        f"{len(analysis_df[analysis_df['category'] == 'Both Wrong'])}"
    ]
}

summary_df = pd.DataFrame(summary_stats)
summary_path = "ai_plagiarism_analysis_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"Summary statistics saved to: {summary_path}")

if len(tfidf_only_df) > 0 and len(codebert_only_df) > 0:
    feature_comparison = {}
    for feature in features_to_compare:
        feature_comparison[feature] = {
            'tfidf_only_mean': float(tfidf_only_df[feature].mean()),
            'codebert_only_mean': float(codebert_only_df[feature].mean()),
            'difference': float(tfidf_only_df[feature].mean() - codebert_only_df[feature].mean())
        }
    
    import json
    feature_path = "ai_plagiarism_feature_comparison.json"
    with open(feature_path, 'w') as f:
        json.dump(feature_comparison, f, indent=2)
    print(f"Feature comparison saved to: {feature_path}")

print("\n" + "="*80)
print("AI PLAGIARISM DETECTION ANALYSIS COMPLETE!")
print("="*80)

plt.show()